# ScribblePrompt-UNet zero-shot - BTXRD (native $128\times128$, scored at original resolution)

No fine-tuning: official pretrained ScribblePrompt-UNet weights, box prompt only
(point/scribble/mask inputs left `None`). Same box protocol as the rest of this
article, reused directly from `PromptSegmentationDataset` in the PGA-UNet
codebase (`_center_zoom_bbox` = covering, `_center_shift_bbox` = off-center),
so the comparison uses exactly the same boxes as every other model.

In [ ]:
# -- Setup -----------------------------------------------------------------
%cd /kaggle/working
import os, gdown, torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# PGA-UNet repo: only needed for its dataset.py (box protocol + polygon listing).
if not os.path.exists('PGA_Unet2D'):
    !git clone --branch main --single-branch https://github.com/ThongLuc2k3/PGA_Unet2D.git
PGA_PATH = '/kaggle/working/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation'

# ScribblePrompt repo (official halleewong implementation) + its checkpoints
if not os.path.exists('ScribblePrompt'):
    !git clone --branch main --single-branch https://github.com/halleewong/ScribblePrompt.git
%cd /kaggle/working/ScribblePrompt
!pip install -q -e .
_have_ckpt = os.path.exists('checkpoints') and any(f.endswith('.pt') for f in os.listdir('checkpoints'))
if not _have_ckpt:
    !bash checkpoints/download.sh
print('checkpoints:', os.listdir('checkpoints') if os.path.exists('checkpoints') else 'MISSING - check download.sh output above')

DATASET_ID   = '1Ep3w8GujpCQ5Qb6nlXKxXhsCxlc3k8gx'
DATASET_ROOT = 'dataset_BTXRD'
if not os.path.exists(f'/kaggle/working/{DATASET_ROOT}'):
    gdown.download(f'https://drive.google.com/uc?id={DATASET_ID}',
                   f'/kaggle/working/{DATASET_ROOT}.zip', quiet=False)
    !unzip -oq /kaggle/working/{DATASET_ROOT}.zip -d /kaggle/working/
print(f'\nSetup complete | dataset={ds_name}')

In [ ]:
# -- Model + shared helpers -------------------------------------------------
import sys, csv, json as _json
import numpy as np
import cv2
import torch
from scipy.ndimage import binary_erosion, distance_transform_edt

if PGA_PATH not in sys.path:
    sys.path.insert(0, PGA_PATH)
from dataset import PromptSegmentationDataset

from scribbleprompt import ScribblePromptUNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DS_NAME = 'BTXRD'
SP_SIZE = 128   # native ScribblePrompt-UNet resolution
TEST_IMG  = f'{PGA_PATH}/{DATASET_ROOT}/test/images'
TEST_JSON = f'{PGA_PATH}/{DATASET_ROOT}/test/annotations'
RESULT_DIR = f'{PGA_PATH}/results'
os.makedirs(RESULT_DIR, exist_ok=True)

sp_model = ScribblePromptUNet()
sp_model = sp_model.to(DEVICE)
if hasattr(sp_model, 'eval'):
    sp_model.eval()

box_ds = PromptSegmentationDataset(TEST_IMG, TEST_JSON, is_train=False, prompt_mode='center_zoom')
print(f'{DS_NAME}: {len(box_ds.all_samples)} (image, polygon) test samples')

def resize_and_pad(arr, size, interp, pad=0):
    h, w = arr.shape[:2]
    sc = min(size / w, size / h)
    nw, nh = max(1, round(w * sc)), max(1, round(h * sc))
    r = cv2.resize(arr, (nw, nh), interpolation=interp)
    out = np.full((size, size), pad, dtype=r.dtype)
    pl, pt = (size - nw) // 2, (size - nh) // 2
    out[pt:pt + nh, pl:pl + nw] = r
    return out, sc, pl, pt, nw, nh

@torch.no_grad()
def scribbleprompt_predict_prob(img_gray, box_xyxy, H, W):
    # img_gray: HxW uint8 at ORIGINAL resolution. box_xyxy: [x0,y0,x1,y1] in
    # ORIGINAL pixel coordinates. Returns a float32 probability map at (H, W).
    canvas, sc, pl, pt, nw, nh = resize_and_pad(img_gray, SP_SIZE, cv2.INTER_LINEAR, 0)
    canvas = canvas.astype(np.float32)
    canvas = (canvas - canvas.min()) / np.clip(canvas.max() - canvas.min(), 1e-8, None)
    img_t = torch.from_numpy(canvas).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

    x0, y0, x1, y1 = box_xyxy
    box_128 = [x0 * sc + pl, y0 * sc + pt, x1 * sc + pl, y1 * sc + pt]
    box_t = torch.as_tensor([[box_128]], dtype=torch.float32, device=DEVICE)   # (B=1, n=1, 4)

    out = sp_model.predict(img_t, None, None, None, box_t, None)   # (1, 1, 128, 128)
    prob_128 = out.squeeze().float().cpu().numpy()

    # Undo pad+resize: crop the valid (non-padded) region, then resize back to (H, W).
    crop = prob_128[pt:pt + nh, pl:pl + nw]
    prob_full = cv2.resize(crop, (W, H), interpolation=cv2.INTER_LINEAR)
    return prob_full

# -- Metrics: identical formulas to the rest of the article --
def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any():
        return 0.0
    S = max(pred.shape)
    if not p.any() or not g.any():
        return float(S)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(S) if not len(d1) or not len(d2) else float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_metrics_img(pred_bin, gt_bin, eps=1e-6):
    pm, gm = pred_bin.astype(np.float32), gt_bin.astype(np.float32)
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    hd95 = calc_hd95(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0:
        cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        diag = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2) / diag, 0, 1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float(tp/(tp+fp+eps)), recall=float(tp/(tp+fn+eps)), hd95=hd95, cbl=cbl)

print('helpers ready | device', DEVICE)
# -- Qualitative visualization: same 5-column style as
# Finetune_SAMMed2D_test_robust.ipynb (Input / Prompt / Prediction / GT / TP-FP-FN),
# 10 shared stems, one PNG + one export_qualitative_rows call per stem.
def visualize_qualitative(by_image_vis, img_dir, box_ds_vis, model_label, prefix_tag):
    import sys as _sys
    from pathlib import Path as _Path
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    if str(_Path(PGA_PATH)) not in _sys.path:
        _sys.path.insert(0, str(_Path(PGA_PATH)))
    from qualitative_visualization import export_qualitative_rows, select_shared_stems

    modes = [('covering', 'pred_zoom', 'limegreen'), ('off-center', 'pred_shift', 'tomato')]
    selection_records = [dict(img_name=n) for n in by_image_vis.keys()]
    vis_images = select_shared_stems(selection_records, n_multi=5, n_single=5)

    for vis_name in vis_images:
        if vis_name not in by_image_vis:
            continue
        rec = by_image_vis[vis_name]
        img_gray = cv2.imread(os.path.join(img_dir, vis_name), cv2.IMREAD_GRAYSCALE)
        H, W = img_gray.shape
        rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
        gt = rec['gt']
        ys, xs = np.where(gt > 0)
        gt_ov = rgb.copy()
        gt_ov[gt > 0] = np.clip(rgb[gt > 0] * 0.4 + np.array([0, 200, 0]) * 0.6, 0, 255)

        fig, axes = plt.subplots(len(modes), 5, figsize=(20, 4 * len(modes)))
        fig.suptitle(f'{model_label}: {vis_name}', fontsize=13, fontweight='bold', y=1.01)
        mode_records = []
        for row, (mode, key, color) in enumerate(modes):
            pred = rec[key].astype(np.uint8)
            pr_ov = rgb.copy()
            pr_ov[pred > 0] = np.clip(rgb[pred > 0] * 0.4 + np.array([220, 60, 60]) * 0.6, 0, 255)
            diff = rgb.copy()
            diff[gt > 0] = [0, 200, 0]
            diff[pred > 0] = [200, 60, 60]
            diff[(gt > 0) & (pred > 0)] = [220, 200, 0]

            axes[row, 0].imshow(img_gray, cmap='gray')
            axes[row, 0].set_ylabel(mode, fontsize=11, fontweight='bold', color=color,
                                    rotation=0, labelpad=65, va='center')
            axes[row, 1].imshow(img_gray, cmap='gray')
            if len(xs) > 0:
                x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
                if mode == 'covering':
                    bx0, bx1, by0, by1 = box_ds_vis._center_zoom_bbox(x0, x1, y0, y1, H, W)
                else:
                    bx0, bx1, by0, by1 = box_ds_vis._center_shift_bbox(x0, x1, y0, y1, H, W, seed_idx=row)
                axes[row, 1].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0,
                                                  linewidth=2.5, edgecolor=color, facecolor='none'))
            axes[row, 2].imshow(pr_ov)
            axes[row, 3].imshow(gt_ov)
            axes[row, 4].imshow(diff)
            mode_records.append(dict(img_name=f'{vis_name}_{mode}', gt=gt.copy(), pred=pred.copy()))
            for ax in axes[row]:
                ax.axis('off')

        plt.tight_layout()
        out = f'{prefix_tag}_{os.path.splitext(vis_name)[0]}.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        export_qualitative_rows(fig, axes, mode_records, prefix=f'{prefix_tag}_{vis_name}')
        print(f'saved {out}')


## Test - 2 prompt modes (image-level merging), zero-shot, no fine-tuning

In [ ]:
# -- Test loop ---------------------------------------------------------------
by_image = {}
names = sorted(set(n for n, _ in box_ds.all_samples))

for n_done, img_name in enumerate(names):
    base = os.path.splitext(img_name)[0]
    img_gray = cv2.imread(os.path.join(TEST_IMG, img_name), cv2.IMREAD_GRAYSCALE)
    H, W = img_gray.shape

    with open(os.path.join(TEST_JSON, base + '.json'), encoding='utf-8') as f:
        data = _json.load(f)

    pred_zoom  = np.zeros((H, W), dtype=np.float32)
    pred_shift = np.zeros((H, W), dtype=np.float32)
    gt_union   = np.zeros((H, W), dtype=np.uint8)

    poly_idxs = [i for i, s in enumerate(data.get('shapes', [])) if s.get('shape_type') == 'polygon']
    for shape_idx in poly_idxs:
        points = np.array(data['shapes'][shape_idx]['points'])
        cv2.fillPoly(gt_union, [points.astype(np.int32)], 1)
        x_min, y_min = points.min(axis=0)
        x_max, y_max = points.max(axis=0)

        sample_idx = box_ds.all_samples.index((img_name, shape_idx))
        bz = box_ds._center_zoom_bbox(x_min, x_max, y_min, y_max, H, W)
        bs = box_ds._center_shift_bbox(x_min, x_max, y_min, y_max, H, W, seed_idx=sample_idx)
        box_zoom  = [bz[0], bz[2], bz[1], bz[3]]
        box_shift = [bs[0], bs[2], bs[1], bs[3]]

        prob_zoom  = scribbleprompt_predict_prob(img_gray, box_zoom, H, W)
        prob_shift = scribbleprompt_predict_prob(img_gray, box_shift, H, W)
        pred_zoom  = np.maximum(pred_zoom, prob_zoom)
        pred_shift = np.maximum(pred_shift, prob_shift)

    by_image[img_name] = dict(pred_zoom=pred_zoom, pred_shift=pred_shift, gt=gt_union)
    if (n_done + 1) % 25 == 0:
        print(f'  {n_done + 1}/{len(names)} images')

print(f'\n{DS_NAME}: {len(by_image)} images processed (zero-shot ScribblePrompt-UNet, no fine-tuning)')

In [ ]:
# -- Image-level merged metrics, both prompt conditions ----------------------
rows = []
for mode, key in [('covering', 'pred_zoom'), ('off-center', 'pred_shift')]:
    per_image = []
    for img_name, rec in by_image.items():
        pred_bin = (rec[key] > 0.5).astype(np.uint8)
        m = calc_metrics_img(pred_bin, rec['gt'])
        m['image'] = img_name
        per_image.append(m)
    agg = {k: float(np.mean([r[k] for r in per_image])) for k in ('dice', 'iou', 'precision', 'recall', 'hd95', 'cbl')}
    rows.append(dict(dataset=DS_NAME, model='ScribblePrompt-UNet (zero-shot)', prompt=mode, **agg))
    with open(f'{RESULT_DIR}/scribbleprompt_zeroshot_{DS_NAME.lower()}_{mode.replace(chr(45),chr(95))}_per_image.csv',
              'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(per_image[0].keys())); w.writeheader(); w.writerows(per_image)

print(f'{DS_NAME}: ScribblePrompt-UNet zero-shot, image-level merged\n')
print(f'{"prompt":<12}{"Dice":>8}{"IoU":>8}{"Prec":>8}{"Recall":>8}{"HD95":>8}{"CBL":>8}')
print('-' * 60)
for r in rows:
    print(f'{r["prompt"]:<12}{r["dice"]:>8.3f}{r["iou"]:>8.3f}{r["precision"]:>8.3f}'
          f'{r["recall"]:>8.3f}{r["hd95"]:>8.1f}{r["cbl"]:>8.3f}')

with open(f'{RESULT_DIR}/scribbleprompt_zeroshot_{DS_NAME.lower()}_summary.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

# -- Qualitative figures (10 shared stems, same style as Finetune_SAMMed2D_test_robust.ipynb) --
visualize_qualitative(by_image, TEST_IMG, box_ds, 'ScribblePrompt-UNet (zero-shot)', 'scribbleprompt_zeroshot_{}'.format(DS_NAME.lower()))
